# Thảo luận, phân tích lỗi và kết luận

## Những kết quả đạt được trên đủ 25.000 bài báo

1. **Phủ trọn dải 25.000 bài của cả 5 thành viên**: sau khi chạy `--resume` M1 cho TRI trên Google Colab CPU-only, tập hợp nhất đạt `25.000` publication duy nhất (`0` trùng lặp chéo thành viên), `41.498` version (`41.487` đã audit xác minh), `976.351` working references (`0` mồ côi) và `456.149` cạnh arXiv trong `course_export`.
2. **Chất lượng parser & khử trùng M2**: trên `38.554` version TeX đã xác minh, parser đạt **85,66% `success`** (`33.025` version) và **97,34% parse được vào cấu trúc** (`37.528` gồm cả `4.503` `partial`), còn `2,66%` (`1.026` version) lỗi; cả `2.764` version `empty_no_tex` trùng khớp với bằng chứng kiểm toán M1. Khử trùng xuyên version giảm `35,04%` số phần tử (`23.099.589 -> 15.006.000`) và `35,83%` số mục BibTeX (`1.834.650 -> 1.177.293`).
3. **Tính hợp lệ 5 bảng & hiệu năng Colab CPU-only**: `validate_project` xác nhận `structural_valid=True` với `0` lỗi khóa/graph trên `22.532.299` cạnh `hierarchy`. Tổng thời gian parse và validate 25.000 bài cold là **18.692,09 giây** (~5 giờ 12 phút, trung bình **0,7477 giây/bài**), đỉnh RSS toàn phiên **845,34 MiB** (< 7% giới hạn 12,67 GiB của Colab Free) và output 3 bảng Parquet chuẩn còn **3,93 GiB** (`4,22 GB`).

## Phân tích nguyên nhân top diagnostics còn lại

Dưới đây là 15 diagnostic phổ biến nhất trên toàn bộ 41.498 version, cho thấy nguyên nhân `processing_complete=false` dù `structural_valid=True`:

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
data = json.loads((Path.cwd().parent / 'report_summary.json').read_text(encoding='utf-8'))
scope, m1, m2, bench = data['scope'], data['m1'], data['m2'], data['m2_benchmark']
plt.rcParams.update({'figure.figsize': (9.2, 4.2), 'axes.spines.top': False, 'axes.spines.right': False, 'figure.dpi': 120})
print(f"Toàn nhóm (5 thành viên): {scope['papers']:,} paper / {scope['versions']:,} version | Cập nhật: {data['report_date']}")

In [ ]:
top_d = pd.DataFrame(m2['top_diagnostics'])
display(top_d)

- **`item outside supported list` (9.087 version)**: phần lớn đến từ macro hoặc gói định nghĩa danh sách/thuật toán không chuẩn ngoài nhóm `itemize`/`enumerate`/`description`/`compactitem`/`inparaenum`/`list`/`trivlist`; parser chuyển trạng thái sang `partial` để giữ lại phần nội dung parse được thay vì bỏ cả version.
- **`Unclosed TeX group` (285 version) và `unmatched end environment: multline` (90 version)**: dấu ngoặc nhọn không cân bằng trong macro hoặc môi trường toán viết tắt từ tác giả làm dừng nhánh con hiện tại.
- **`dynamic/unsupported include in main.tex` (164 version) và `missing/unsafe bibliography: references.bib` (57 version)**: macro truyền đường dẫn include động không phải chuỗi tĩnh, hoặc tệp bibliography khai báo trong `.tex` không có mặt trong gói nguồn.
- **11 version `source_pending_or_failed` của TRIET từ M1** và **169 version HTTP 404** đã xác minh: được ghi là `error_missing_source` (`180` version), không tạo phần tử giả.

## Kết luận và hướng cho Milestone 3

Bộ 5 bảng Parquet (`papers`, `references`, `elements`, `hierarchy`, `bibtex`) của toàn bộ **25.000 bài báo** đã hợp lệ về cấu trúc và khóa ngoại, đủ điều kiện làm đầu vào cho bài toán Reference Matching ở Milestone 3.

## Tài liệu và mã nguồn tham chiếu

- [Đặc tả chính thức M2](https://hcmus-cs-ta.github.io/lab-book/intro-ds/statement/parsing/).
- [arXiv API User's Manual](https://info.arxiv.org/help/api/user-manual.html) & [Terms of Use](https://info.arxiv.org/help/api/tou.html).
- [Semantic Scholar Academic Graph API](https://api.semanticscholar.org/api-docs/graph).
- Mã nguồn triển khai: [src/parser/validation.py](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/src/parser/validation.py).
- Mã nguồn triển khai: [tools/m2_team_consolidator.py](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/tools/m2_team_consolidator.py).
- Mã nguồn triển khai: [tools/build_team_25000_book.py](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/tools/build_team_25000_book.py).